### Feature Engineering

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel,RFE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix

cc_data = pd.read_csv("credit_card_clean.csv")

In [2]:
del cc_data["ID"]

In [3]:
encoded = LabelEncoder()
cc_data["SEX"] = encoded.fit_transform(cc_data["SEX"])
cc_data["EDUCATION"] = encoded.fit_transform(cc_data["EDUCATION"])
cc_data["MARRIAGE"] = encoded.fit_transform(cc_data["MARRIAGE"])

In [4]:
X =cc_data.drop("DEFAULT",axis=1)
y = cc_data["DEFAULT"]

In [5]:
X.columns

Index(['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1', 'PAY_2',
       'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'],
      dtype='object')

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=.8,random_state=30,shuffle=True,stratify=y)

In [7]:
selector = SelectFromModel(estimator=RandomForestClassifier())
selector.fit(X_train,y_train)


,estimator,RandomForestClassifier()
,threshold,None
,prefit,False
,norm_order,1
,max_features,None
,importance_getter,'auto'
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1


In [8]:
X_train_selecter =selector.transform(X_train)
X_test_selecter =selector.transform(X_test)

In [9]:
len(X.columns[selector.get_support()])

15

In [10]:
def randomForestModel(X_train, X_test, y_train, y_test):
    rf_model = RandomForestClassifier(max_depth=7,random_state=31)
    
    rf_model.fit(X_train,y_train)
    y_pred_test = rf_model.predict(X_test)
    
    print("Accuracy Score",accuracy_score(y_test,y_pred_test))
    print("Precision",precision_score(y_test,y_pred_test))
    print("Recall",recall_score(y_test,y_pred_test))
    print("confusion Martix \n",confusion_matrix(y_test,y_pred_test))

In [11]:
%%time
randomForestModel(X_train, X_test, y_train, y_test)

Accuracy Score 0.8245
Precision 0.7082066869300911
Recall 0.35116804822908815
confusion Martix 
 [[4481  192]
 [ 861  466]]
CPU times: total: 2.61 s
Wall time: 2.62 s


In [12]:
%%time
randomForestModel(X_train_selecter, X_test_selecter, y_train, y_test)

Accuracy Score 0.822
Precision 0.7191201353637902
Recall 0.3202712886209495
confusion Martix 
 [[4507  166]
 [ 902  425]]
CPU times: total: 2.53 s
Wall time: 2.54 s


In [13]:
eleminator = RFE(estimator=RandomForestClassifier(),n_features_to_select=6)
eleminator.fit(X_train,y_train)

,estimator,RandomForestClassifier()
,n_features_to_select,6
,step,1
,verbose,0
,importance_getter,'auto'
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [14]:
X_train_eleminator =eleminator.transform(X_train)
X_test_eleminator =eleminator.transform(X_test)

In [15]:
%%time
randomForestModel(X_train_eleminator, X_test_eleminator, y_train, y_test)

Accuracy Score 0.8238333333333333
Precision 0.7368421052631579
Recall 0.3165033911077619
confusion Martix 
 [[4523  150]
 [ 907  420]]
CPU times: total: 2.25 s
Wall time: 2.24 s


### Take the Important Features from GradientBoosting and train it within RandomForest Algorithm

In [16]:
eleminator = RFE(estimator=GradientBoostingClassifier() , n_features_to_select=8)
eleminator.fit(X_train,y_train)

,estimator,GradientBoostingClassifier()
,n_features_to_select,8
,step,1
,verbose,0
,importance_getter,'auto'
,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2


In [17]:
X_train_eleminator = eleminator.transform(X_train)
X_test_eleminator = eleminator.transform(X_test)

In [18]:
%%time
randomForestModel(X_train_eleminator, X_test_eleminator, y_train, y_test)

Accuracy Score 0.8265
Precision 0.7096774193548387
Recall 0.36473247927656366
confusion Martix 
 [[4475  198]
 [ 843  484]]
CPU times: total: 1.42 s
Wall time: 1.4 s
